In [1]:
import pandas as pd
import sqlite3
import os

# ── Paths ──────────────────────────────────────────────
CSV_PATH = "data/telco_churn.csv"
DB_PATH  = "data/churn.db"

# ── Load CSV ───────────────────────────────────────────
df = pd.read_csv(CSV_PATH)

# ── Light cleaning before loading ─────────────────────
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(0)

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

df["churn"] = df["churn"].map({"Yes": 1, "No": 0})

# ── Write to SQLite ────────────────────────────────────
conn = sqlite3.connect(DB_PATH)
df.to_sql("customers", conn, if_exists="replace", index=False)
conn.close()

print(f"✓ Database created at: {DB_PATH}")
print(f"✓ Rows loaded: {len(df):,}")
print(f"✓ Columns: {list(df.columns)}")

✓ Database created at: data/churn.db
✓ Rows loaded: 7,043
✓ Columns: ['customerid', 'gender', 'seniorcitizen', 'partner', 'dependents', 'tenure', 'phoneservice', 'multiplelines', 'internetservice', 'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport', 'streamingtv', 'streamingmovies', 'contract', 'paperlessbilling', 'paymentmethod', 'monthlycharges', 'totalcharges', 'churn']


In [3]:
#Verify data loaded correctly with sanity check
conn = sqlite3.connect("data/churn.db")

# Row count
print(pd.read_sql("SELECT COUNT(*) AS total_rows FROM customers", conn))

# Churn distribution
print(pd.read_sql("""
    SELECT churn,
           COUNT(*)                        AS count,
           ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM customers), 1) AS pct
    FROM customers
    GROUP BY churn
""", conn))

# Check for nulls in key columns
print(pd.read_sql("""
    SELECT
        SUM(CASE WHEN tenure       IS NULL THEN 1 ELSE 0 END) AS tenure_nulls,
        SUM(CASE WHEN totalcharges IS NULL THEN 1 ELSE 0 END) AS totalcharges_nulls,
        SUM(CASE WHEN churn        IS NULL THEN 1 ELSE 0 END) AS churn_nulls
    FROM customers
""", conn))

conn.close

   total_rows
0        7043
   churn  count   pct
0      0   5174  73.5
1      1   1869  26.5
   tenure_nulls  totalcharges_nulls  churn_nulls
0             0                   0            0


<function Connection.close()>